# Four-Currency Cross-Rate Forecasting + Explainable AI

## Faculty Demo — Code → Output → Graph → Analysis

### Only four currencies
- EUR — Euro
- CHF — Swiss Franc
- JPY — Japanese Yen
- KRW — South Korean Won

**No USD. No INR.**

Because there is no external base currency, the practical market analysis uses all six direct cross-rates among the four currencies:

**EUR/CHF, EUR/JPY, EUR/KRW, CHF/JPY, CHF/KRW, JPY/KRW**

The flow is:

**Real market data → Feature engineering → Models → Numerical output → Graphs → SHAP → Analysis**.


## 1. Load real cross-currency market data

The six cross-rates are downloaded in parallel. No USD or INR series is used.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ThreadPoolExecutor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import shap


# ============================================================
# 1. LOAD REAL FX CROSS-RATE DATA
# ============================================================

PAIRS = {
    "EUR/CHF": "EURCHF=X",
    "EUR/JPY": "EURJPY=X",
    "EUR/KRW": "EURKRW=X",
    "CHF/JPY": "CHFJPY=X",
    "CHF/KRW": "CHFKRW=X",
    "JPY/KRW": "JPYKRW=X",
}

FEATURES = [
    "return_1m",
    "momentum_3m",
    "momentum_6m",
    "volatility_3m",
    "volatility_6m",
    "trend_12m",
    "ma_gap",
    "drawdown_12m",
]


def download_pair(item):
    pair, ticker = item

    df = yf.download(
        ticker,
        start=(pd.Timestamp.today() - pd.DateOffset(years=10)).strftime("%Y-%m-%d"),
        end=pd.Timestamp.today().strftime("%Y-%m-%d"),
        interval="1mo",
        auto_adjust=False,
        progress=False,
    )

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    if "Close" not in df.columns:
        raise ValueError(f"Close column not available for {pair}")

    return pair, df[["Close"]].rename(columns={"Close": pair}).dropna()


# Download all six cross-rates in parallel
with ThreadPoolExecutor(max_workers=6) as pool:
    downloaded = list(pool.map(download_pair, PAIRS.items()))

prices = pd.concat(
    [item[1] for item in downloaded],
    axis=1
).sort_index()

print("DATASET SUMMARY")
print("-" * 70)
print("Rows:", len(prices))
print("Date range:", prices.index.min().date(), "to", prices.index.max().date())
print("Currencies used: EUR, CHF, JPY, KRW")
print("USD used: NO")
print()
print("Cross-rates:")
for pair in PAIRS:
    print("•", pair)

print()
print(prices.tail())




### First graph

**EUR, CHF, JPY and KRW cross-currency market movement**

Each cross-rate is indexed to 100 at its first observation so rates with different numerical scales can be compared visually.


In [ ]:
# ============================================================
# GRAPH 1 — SIX CROSS-RATE MOVEMENTS
# ============================================================

fig, axes = plt.subplots(
    2, 3,
    figsize=(16, 8)
)

for ax, pair in zip(axes.ravel(), prices.columns):

    indexed = (
        prices[pair]
        / prices[pair].iloc[0]
        * 100
    )

    ax.plot(
        indexed.index,
        indexed,
        linewidth=2
    )

    ax.set_title(pair)
    ax.set_ylabel("Indexed rate")
    ax.grid(alpha=0.25)

plt.suptitle(
    "EUR, CHF, JPY and KRW — Cross-Currency Market Movement"
)

plt.tight_layout()
plt.show()




## 2. Feature engineering

For every cross-rate, the code calculates:

- 1-month return
- 3-month momentum
- 6-month momentum
- 3-month volatility
- 6-month volatility
- 12-month trend
- moving-average gap
- 12-month drawdown

The target is the future 3-month or 6-month cross-rate return.


In [ ]:
# ============================================================
# 2. FEATURE ENGINEERING
# ============================================================

def create_features(series, horizon_months):

    d = pd.DataFrame({
        "close": series.copy()
    })

    d["return_1m"] = (
        d["close"].pct_change(1)
    )

    d["momentum_3m"] = (
        d["close"].pct_change(3)
    )

    d["momentum_6m"] = (
        d["close"].pct_change(6)
    )

    monthly_return = (
        d["close"].pct_change()
    )

    d["volatility_3m"] = (
        monthly_return.rolling(3).std()
    )

    d["volatility_6m"] = (
        monthly_return.rolling(6).std()
    )

    d["trend_12m"] = (
        d["close"].pct_change(12)
    )

    moving_average = (
        d["close"].rolling(6).mean()
    )

    d["ma_gap"] = (
        d["close"] / moving_average - 1
    )

    rolling_high = (
        d["close"].rolling(12).max()
    )

    d["drawdown_12m"] = (
        d["close"] / rolling_high - 1
    )

    # Future return target
    d[f"target_{horizon_months}m"] = (
        d["close"].shift(-horizon_months)
        / d["close"]
        - 1
    )

    return d.dropna()


print("\nFEATURE ENGINEERING")
print("-" * 70)

print(
    "Raw cross-rate -> derived market features:"
)

for feature in FEATURES:
    print("•", feature)


# ============================================================
# 3. MODELS
# ============================================================

def evaluate_model(
    model,
    train_x,
    train_y,
    test_x,
    test_y
):

    model.fit(
        train_x,
        train_y
    )

    prediction = model.predict(
        test_x
    )

    return {
        "RMSE":
            mean_squared_error(
                test_y,
                prediction
            ) ** 0.5,

        "MAE":
            mean_absolute_error(
                test_y,
                prediction
            ),

        "R2":
            r2_score(
                test_y,
                prediction
            ),

        "Directional Accuracy":
            np.mean(
                np.sign(test_y)
                ==
                np.sign(prediction)
            ),

        "model":
            model,

        "prediction":
            prediction
    }


def run_pair(
    pair,
    horizon
):

    data = create_features(
        prices[pair],
        horizon
    )

    # Chronological 80/20 split
    split = int(
        len(data) * 0.80
    )

    train = data.iloc[:split]
    test = data.iloc[split:]

    X_train = train[FEATURES]
    y_train = train[
        f"target_{horizon}m"
    ]

    X_test = test[FEATURES]
    y_test = test[
        f"target_{horizon}m"
    ]

    # Random-walk zero-return baseline
    rw_prediction = np.zeros(
        len(test)
    )

    results = {
        "Random Walk": {
            "RMSE":
                mean_squared_error(
                    y_test,
                    rw_prediction
                ) ** 0.5,

            "MAE":
                mean_absolute_error(
                    y_test,
                    rw_prediction
                ),

            "R2":
                r2_score(
                    y_test,
                    rw_prediction
                ),

            "Directional Accuracy":
                np.mean(
                    np.sign(y_test)
                    == 0
                ),

            "model": None,

            "prediction":
                rw_prediction
        }
    }

    models = {

        "Ridge":
            Ridge(
                alpha=1.0
            ),

        "Elastic Net":
            ElasticNet(
                alpha=0.001,
                l1_ratio=0.5,
                max_iter=10000
            ),

        "XGBoost":
            XGBRegressor(
                n_estimators=400,
                max_depth=3,
                learning_rate=0.03,
                subsample=0.85,
                colsample_bytree=0.85,
                objective="reg:squarederror",
                random_state=42,
                n_jobs=2
            )
    }

    for name, model in models.items():

        results[name] = evaluate_model(
            model,
            X_train,
            y_train,
            X_test,
            y_test
        )

    return (
        data,
        train,
        test,
        results
    )


all_results = {}
metric_rows = []

for pair in PAIRS:

    for horizon in [3, 6]:

        data, train, test, results = (
            run_pair(
                pair,
                horizon
            )
        )

        all_results[
            (pair, horizon)
        ] = (
            data,
            train,
            test,
            results
        )

        for model_name, values in (
            results.items()
        ):

            metric_rows.append({

                "Cross Pair":
                    pair,

                "Horizon":
                    f"{horizon}-month",

                "Model":
                    model_name,

                "RMSE":
                    values["RMSE"],

                "MAE":
                    values["MAE"],

                "R2":
                    values["R2"],

                "Directional Accuracy":
                    values[
                        "Directional Accuracy"
                    ]
            })


metrics = pd.DataFrame(
    metric_rows
)

print("\nMODEL RESULTS")
print("-" * 70)

print(
    metrics.round(4)
    .to_string(index=False)
)




## 3. Run the models

For every cross-rate and both horizons:

- Random Walk
- Ridge
- Elastic Net
- XGBoost

Output:

- RMSE
- MAE
- R²
- Directional Accuracy


In [ ]:
# ============================================================
# GRAPH 2 — MODEL COMPARISON
# ============================================================

for horizon in [3, 6]:

    subset = metrics[
        metrics["Horizon"]
        == f"{horizon}-month"
    ]

    plt.figure(
        figsize=(13, 6)
    )

    sns.barplot(
        data=subset,
        x="Cross Pair",
        y="RMSE",
        hue="Model"
    )

    plt.title(
        f"Model Comparison — {horizon}-Month Cross-Rate Forecast"
    )

    plt.ylabel(
        "RMSE — lower is better"
    )

    plt.xlabel(
        "Cross Currency Pair"
    )

    plt.grid(
        axis="y",
        alpha=0.25
    )

    plt.tight_layout()
    plt.show()




In [ ]:
# ============================================================
# GRAPH 3 — ACTUAL VS XGBOOST
# ============================================================

for pair in PAIRS:

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(15, 5)
    )

    for ax, horizon in zip(
        axes,
        [3, 6]
    ):

        data, train, test, results = (
            all_results[
                (pair, horizon)
            ]
        )

        actual = test[
            f"target_{horizon}m"
        ]

        prediction = results[
            "XGBoost"
        ]["prediction"]

        ax.plot(
            test.index,
            actual * 100,
            linewidth=2,
            label="Actual"
        )

        ax.plot(
            test.index,
            prediction * 100,
            linewidth=2,
            label="XGBoost"
        )

        ax.axhline(
            0,
            linewidth=0.8
        )

        ax.set_title(
            f"{pair} — {horizon}-Month Forecast"
        )

        ax.set_ylabel(
            "Future Return (%)"
        )

        ax.legend()

        ax.grid(
            alpha=0.25
        )

    plt.suptitle(
        f"Actual vs XGBoost Predicted Cross-Rate Returns — {pair}"
    )

    plt.tight_layout()
    plt.show()




### How to explain the graphs

The model-comparison graphs compare forecasting error across the six cross-rates.

The actual-vs-predicted graphs show how the XGBoost prediction follows the observed future cross-rate return.

The 3-month and 6-month horizons allow a direct comparison of short- and longer-horizon forecasting behavior.


## 4. SHAP explainability

SHAP identifies which engineered market features contributed most to the XGBoost predictions.

It explains model behavior; it does not establish causality.


In [ ]:
# ============================================================
# 4. SHAP EXPLAINABILITY
# ============================================================

shap_importance = {}

for pair in PAIRS:

    data, train, test, results = (
        all_results[
            (pair, 6)
        ]
    )

    model = results[
        "XGBoost"
    ]["model"]

    explainer = shap.TreeExplainer(
        model
    )

    shap_values = (
        explainer.shap_values(
            test[FEATURES]
        )
    )

    importance = (
        pd.Series(
            np.abs(
                shap_values
            ).mean(axis=0),
            index=FEATURES
        )
        .sort_values(
            ascending=False
        )
    )

    shap_importance[
        pair
    ] = importance

    plt.figure(
        figsize=(9, 5)
    )

    importance.sort_values().plot(
        kind="barh"
    )

    plt.title(
        f"SHAP Feature Importance — 6-Month {pair} XGBoost"
    )

    plt.xlabel(
        "Mean |SHAP value|"
    )

    plt.tight_layout()
    plt.show()


# ============================================================
# GRAPH 4 — SHAP COMPARISON
# ============================================================

shap_table = pd.DataFrame(
    shap_importance
)

plt.figure(
    figsize=(13, 7)
)

sns.heatmap(
    shap_table,
    annot=True,
    fmt=".4f"
)

plt.title(
    "SHAP Feature Importance Across EUR, CHF, JPY and KRW Cross-Rates"
)

plt.xlabel(
    "Cross Currency Pair"
)

plt.ylabel(
    "Feature"
)

plt.tight_layout()
plt.show()


print("\nSHAP IMPORTANCE")
print("-" * 70)

print(
    shap_table.round(4)
)




## 5. Final analysis

The final cell automatically reports the lowest-RMSE model, highest-R² model and strongest SHAP feature for each cross-rate.

### Presentation sequence

**Code → Run → Numerical Output → Graph → Interpretation**

This version contains **only EUR, CHF, JPY and KRW**. There is no USD or INR in the data, code, graphs or analysis.


In [ ]:
# ============================================================
# 5. FACULTY-FRIENDLY ANALYSIS WITH NUMERICAL VALUES
# ============================================================

print("\nFACULTY ANALYSIS — NUMERICAL RESULTS")
print("-" * 80)

for pair in PAIRS:

    print(f"\n{pair}")

    for horizon in [3, 6]:

        subset = metrics[
            (metrics["Cross Pair"] == pair) &
            (metrics["Horizon"] == f"{horizon}-month")
        ].copy()

        # Model with the lowest RMSE
        rmse_row = subset.loc[subset["RMSE"].idxmin()]

        # Model with the highest R2
        r2_row = subset.loc[subset["R2"].idxmax()]

        # Model with the highest directional accuracy
        accuracy_subset = subset.dropna(
            subset=["Directional Accuracy"]
        )
        accuracy_row = accuracy_subset.loc[
            accuracy_subset["Directional Accuracy"].idxmax()
        ]

        print(
            f"{horizon}-month:"
        )

        print(
            f"  Lowest RMSE       = {rmse_row['RMSE']:.6f} "
            f"({rmse_row['Model']})"
        )

        print(
            f"  Highest R2        = {r2_row['R2']:.6f} "
            f"({r2_row['Model']})"
        )

        print(
            f"  Lowest MAE        = "
            f"{subset.loc[subset['MAE'].idxmin(), 'MAE']:.6f}"
        )

        print(
            f"  Highest Accuracy  = "
            f"{accuracy_row['Directional Accuracy'] * 100:.2f}% "
            f"({accuracy_row['Model']})"
        )

    top_feature = shap_importance[pair].index[0]
    top_shap_value = shap_importance[pair].iloc[0]

    print(
        f"  6-month XGBoost strongest SHAP feature = "
        f"{top_feature} ({top_shap_value:.6f})"
    )


print("\nImportant:")
print("Only EUR, CHF, JPY and KRW are used.")
print("No USD or INR is used anywhere in the analysis.")
print("RMSE and MAE are error measures; lower values indicate smaller prediction errors.")
print("R2 measures explained variance; higher values indicate better fit.")
print("Directional Accuracy shows how often the predicted return direction matches the actual direction.")
print("SHAP explains model contribution; it does not prove causality.")
print("The chronological train/test split helps avoid future-data leakage.")
